# Step 10c-2: gpt-4o-mini chấm 4 bộ mới (SciFact, NFCorpus, DBPedia-entity, FEVER)

| Khối | Chế độ | Tập | Số lượt gọi | Dùng cho |
|---|---|---|---|---|
| A | `paper`, `dat` | test của cả 4 bộ | 2 × 7.689 | bảng chính (QuDAR-llm, DAT) |
| B | `code`, `paper_fixed`, `code_shuffle` | test SciFact, NFCorpus, DBPedia-entity; 1.500 query đầu của FEVER test | 3 × 2.523 | ablation 2×2, thiên lệch vị trí |
| C | `paper` | train SciFact, NFCorpus, 10.000 query train đầu của FEVER | 13.399 | mô hình học trọng số có đặc trưng LLM (như FiQA, HotpotQA ở Step 4) |

- **Accelerator:** None (CPU). **Internet:** On. **Secret:** `OPENAI_API_KEY` 
- **Input:** `qudar-beir-data-2`, `qudar-runs-2`, `qudar-patches`
- **Chi phí ước tính:** khoảng 5 USD (giá 0,15 / 0,60 USD mỗi triệu token vào / ra). Notebook tự bỏ qua các lượt gọi còn lại khi chi phí ước tính vượt `BUDGET_USD = 10` (sửa ở ô 1 nếu cần)
- **Thời gian ước tính:** khối A khoảng 1,5 giờ, B khoảng 45 phút, C khoảng 1,5 giờ, cộng 15–20 phút đọc file FEVER. **Tổng 3,5–4,5 giờ** 
- **Cách chạy:** Save Version → **Save & Run All (Commit)**. Ô 2 gọi thử 5 query trước; nếu key sai hoặc API lỗi, notebook dừng ngay ở đó.

In [1]:
import glob, json, os
from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "OPENAI_API_KEY phải là secret key dạng sk-..."  # không in key

def first(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"không tìm thấy {what} ({pattern}): kiểm tra dataset đã gắn chưa"
    return hits[0]
DATA2 = os.path.dirname(os.path.dirname(first("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", "qudar-beir-data-2")))
PATCH = os.path.dirname(first("/kaggle/input/**/llm_score.py", "qudar-patches"))
RUNS  = os.path.dirname(os.path.dirname(first("/kaggle/input/**/fever_test/retrieval_result.json", "qudar-runs-2")))
TEST  = ["scifact_test", "nfcorpus_test", "dbpedia-entity_test", "fever_test"]
TRAIN = ["scifact_train", "nfcorpus_train", "fever_train"]
missing = [n for n in TEST + TRAIN if not os.path.exists(f"{RUNS}/{n}/retrieval_result.json")]
assert not missing, f"qudar-runs-2 thiếu: {missing}"
assert "--max-queries" in open(f"{PATCH}/llm_score.py").read(), "qudar-patches chưa phải bản mới"
print("DATA2 =", DATA2, "\nPATCH =", PATCH, "\nRUNS  =", RUNS)
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
!pip install -q -e /kaggle/working/QuDAR
OUT = "/kaggle/working/llm_gpt"
os.makedirs(OUT, exist_ok=True)
L = f"python {PATCH}/llm_score.py --repo /kaggle/working/QuDAR --runs {RUNS} --data-root {DATA2} --out {OUT} --workers 3"

PRICE_IN, PRICE_OUT = 0.15, 0.60   # USD mỗi 1 triệu token của gpt-4o-mini
OUT_TOKENS = 12                    # số token trả lời trung bình mỗi lượt, ví dụ "[3,4,5,1]"
BUDGET_USD = 10.0                   # vượt mức này thì các lượt sau bị bỏ qua, không gọi API nữa

def spent():
    """Chi phí ước tính từ cache: cột tokens là tổng token vào + ra của mỗi lượt."""
    tok = calls = 0
    for f in glob.glob(f"{OUT}/llm_*.jsonl"):
        for line in open(f):
            tok += json.loads(line).get("tokens", 0)
            calls += 1
    return (tok * PRICE_IN + calls * OUT_TOKENS * (PRICE_OUT - PRICE_IN)) / 1e6

def run(modes, names, extra=""):
    for n in names:
        s = spent()
        if s > BUDGET_USD:
            print(f"BỎ QUA {n} [{modes}]: đã tiêu khoảng {s:.2f} USD > BUDGET_USD = {BUDGET_USD}")
            continue
        print(f"=== {n} [{modes}] {extra} | đã tiêu khoảng {s:.2f} USD", flush=True)
        get_ipython().system(f"{L} --mode {modes} --names {n} {extra}")

DATA2 = /kaggle/input/datasets/karrychuong/qudar-beir-data-2/data 
PATCH = /kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches 
RUNS  = /kaggle/input/datasets/karrychuong/qudar-runs-2/runs
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for qudar (pyproject.toml) ... done


In [2]:
# Gọi thử 5 query SciFact (chế độ paper). Cache được dùng lại ở khối A nên không tốn thêm.
run("paper", ["scifact_test"], "--max-queries 5")
cache = f"{OUT}/llm_paper_scifact_test.jsonl"
rows = [json.loads(l) for l in open(cache)] if os.path.exists(cache) else []
ok = sum(r["scores"] is not None for r in rows)
print(f"{ok}/{len(rows)} query chấm được, trung bình {sum(r['tokens'] for r in rows) / max(len(rows), 1):.0f} token mỗi lượt")
assert ok >= 4, "Gọi API thất bại: xem log ở trên (key, số dư, tên model). Notebook dừng ở đây, chưa tốn tiền đáng kể."

=== scifact_test [paper] --max-queries 5 | đã tiêu khoảng 0.00 USD
scifact_test [paper]: 5 queries, 0 cached, 5 to score
  5/5 scored, 56/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens/query 1316
LLM fusion: 100%|████████████████████████████████| 5/5 [00:00<00:00, 951.35it/s]
               name    method  Recall@10  nDCG@10  Precision@1
scifact_test_first5 qudar_llm        0.5   0.2159          0.0
5/5 query chấm được, trung bình 1316 token mỗi lượt


In [3]:
# A. Bảng chính: QuDAR-llm (prompt paper) và DAT trên test của cả 4 bộ, bộ nhỏ trước
run("paper dat", TEST)
print(f"Sau khối A: khoảng {spent():.2f} USD")

=== scifact_test [paper dat]  | đã tiêu khoảng 0.00 USD
scifact_test [paper]: 300 queries, 5 cached, 295 to score
  295/295 scored, 192/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens/query 1298
LLM fusion: 100%|████████████████████████████| 300/300 [00:00<00:00, 934.76it/s]
        name    method  Recall@10  nDCG@10  Precision@1
scifact_test qudar_llm     0.8643   0.7226       0.5733
scifact_test [dat]: 300 queries, 0 cached, 300 to score
  300/300 scored, 298/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens/query 777
LLM fusion: 100%|███████████████████████████| 300/300 [00:00<00:00, 1039.42it/s]
        name method  Recall@10  nDCG@10  Precision@1
scifact_test    dat     0.8551   0.7211       0.5867
=== nfcorpus_test [paper dat]  | đã tiêu khoảng 0.10 USD
nfcorpus_test [paper]: 323 queries, 0 cached, 323 to score
  323/323 scored, 202/min, ~0 min left
nfcorpus_test: 0 failed (equal weights used), avg tokens/query 1172
LLM fusion: 100%

In [4]:
# B. Ablation 2x2 (prompt paper/code × xáo/cố định). FEVER lấy 1.500 query đầu như HotpotQA ở Step 6;
#    chế độ paper trên tập con lấy từ cache của khối A nên không tốn thêm.
run("code paper_fixed code_shuffle", TEST[:3])
run("paper code paper_fixed code_shuffle", ["fever_test"], "--max-queries 1500")
print(f"Sau khối B: khoảng {spent():.2f} USD")

=== scifact_test [code paper_fixed code_shuffle]  | đã tiêu khoảng 1.56 USD
scifact_test [code]: 300 queries, 0 cached, 300 to score
  300/300 scored, 241/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens/query 1218
LLM fusion: 100%|████████████████████████████| 300/300 [00:00<00:00, 630.26it/s]
        name               method  Recall@10  nDCG@10  Precision@1
scifact_test qudar_llm_codeprompt     0.8676   0.7346       0.5933
scifact_test [paper_fixed]: 300 queries, 0 cached, 300 to score
  300/300 scored, 193/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens/query 1298
LLM fusion: 100%|████████████████████████████| 300/300 [00:00<00:00, 763.96it/s]
        name                method  Recall@10  nDCG@10  Precision@1
scifact_test qudar_llm_paper_fixed     0.8684   0.7294       0.5833
scifact_test [code_shuffle]: 300 queries, 0 cached, 300 to score
  300/300 scored, 251/min, ~0 min left
scifact_test: 0 failed (equal weights used), avg tokens

In [5]:
# C. Tập train (chế độ paper) cho mô hình học trọng số có đặc trưng LLM
run("paper", TRAIN)
print(f"Sau khối C: khoảng {spent():.2f} USD")

=== scifact_train [paper]  | đã tiêu khoảng 2.51 USD
scifact_train [paper]: 809 queries, 0 cached, 809 to score
  500/809 scored, 191/min, ~2 min left
  809/809 scored, 192/min, ~0 min left
scifact_train: 0 failed (equal weights used), avg tokens/query 1321
LLM fusion: 100%|████████████████████████████| 809/809 [00:01<00:00, 769.25it/s]
         name    method  Recall@10  nDCG@10  Precision@1
scifact_train qudar_llm     0.8603   0.7321       0.6082
=== nfcorpus_train [paper]  | đã tiêu khoảng 2.68 USD
nfcorpus_train [paper]: 2590 queries, 0 cached, 2590 to score
  500/2590 scored, 187/min, ~11 min left
  1000/2590 scored, 188/min, ~8 min left
  1500/2590 scored, 190/min, ~6 min left
  2000/2590 scored, 192/min, ~3 min left
  2500/2590 scored, 193/min, ~0 min left
  2590/2590 scored, 194/min, ~0 min left
nfcorpus_train: 0 failed (equal weights used), avg tokens/query 1210
LLM fusion: 100%|██████████████████████████| 2590/2590 [00:05<00:00, 457.38it/s]
          name    method  Recall@10

In [6]:
# Tóm tắt + đóng gói cho repo
import random, shutil
import numpy as np, pandas as pd
from pathlib import Path
pd.set_option("display.width", 250)

ev = pd.read_csv(f"{OUT}/llm_eval.csv").drop_duplicates(["name", "method"], keep="last")
ev = ev[~ev["name"].str.endswith("_first5")]
lab = {"qudar_llm": "llm paper", "qudar_llm_codeprompt": "llm code", "qudar_llm_paper_fixed": "paper cố định",
       "qudar_llm_code_shuffle": "code xáo", "dat": "DAT"}
cells = {}
for _, r in ev.iterrows():
    cells.setdefault(r["name"], {})[lab[r["method"]]] = f'{r["Recall@10"]:.4f} / {r["nDCG@10"]:.4f}'
for name in cells:
    if "_first" not in name:  # Equal trên tập con chưa có, sẽ tính ở bước kiểm định
        eq = pd.read_csv(f"{RUNS}/{name}/eval_results.csv").set_index("method").loc["qudar_simple_equal"]
        cells[name] = {"Equal": f'{eq["Recall@10"]:.4f} / {eq["nDCG@10"]:.4f}', **cells[name]}
cols = ["Equal", "llm paper", "llm code", "paper cố định", "code xáo", "DAT"]
tab = pd.DataFrame.from_dict(cells, orient="index").reindex(columns=cols)
print("Recall@10 / nDCG@10 (Equal lấy từ Step 10c-1, không dùng LLM)")
print(tab.fillna("").to_string())
bad = ev[ev["failed"] > 0]
print("\nQuery chấm lỗi (dùng trọng số đều):", "không có" if bad.empty else "\n" + bad[["name", "method", "queries", "failed"]].to_string(index=False))

# Điểm trung bình theo vị trí hiển thị (cùng cách tính với Step 10-B1)
rows = []
for f in sorted(Path(OUT).glob("llm_*.jsonl")):
    mode = next((m for m in ("paper_fixed", "code_shuffle", "paper", "code") if f.name.startswith(f"llm_{m}_")), None)
    if mode is None:
        continue
    pos = {i: [] for i in range(1, 5)}
    for line in f.read_text().splitlines():
        r = json.loads(line)
        if not r["scores"]:
            continue
        order = ["ES", "OS", "ED", "OD"]
        if mode in ("paper", "code_shuffle"):
            order = ["OS", "OD", "ES", "ED"]
            random.Random(r["qid"]).shuffle(order)
        for i, s in enumerate(order, 1):
            pos[i].append(r["scores"][s])
    rows.append({"file": f.stem, "n": len(pos[1]), **{f"pos {i}": round(float(np.mean(v)), 2) for i, v in pos.items()}})
print("\n" + pd.DataFrame(rows).to_string(index=False))
print(f"\nChi phí ước tính cả notebook: khoảng {spent():.2f} USD (đối chiếu với trang Usage của OpenAI)")

dst = Path("/kaggle/working/for_repo/results/step10/llm_gpt")
shutil.copytree(OUT, dst, dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/for_repo_step10c2", "zip", "/kaggle/working/for_repo")
!du -sh {OUT} /kaggle/working/for_repo_step10c2.zip

Recall@10 / nDCG@10 (Equal lấy từ Step 10c-1, không dùng LLM)
                                Equal        llm paper         llm code    paper cố định         code xáo              DAT
scifact_test          0.8649 / 0.7231  0.8643 / 0.7226  0.8676 / 0.7346  0.8684 / 0.7294  0.8659 / 0.7287  0.8551 / 0.7211
nfcorpus_test         0.1726 / 0.3528  0.1756 / 0.3548  0.1757 / 0.3592  0.1763 / 0.3593  0.1770 / 0.3600  0.1592 / 0.3305
dbpedia-entity_test   0.2595 / 0.3978  0.2466 / 0.3962  0.2530 / 0.4086  0.2518 / 0.4030  0.2494 / 0.4050  0.2170 / 0.3374
fever_test            0.7901 / 0.6357  0.7871 / 0.6520                                                     0.6651 / 0.5053
scifact_train         0.8500 / 0.7168  0.8603 / 0.7321                                                                    
nfcorpus_train        0.1621 / 0.3435  0.1649 / 0.3505                                                                    
fever_train           0.7563 / 0.5876  0.7552 / 0.6060                       